# Visual / DSP Ablation Experiment — Colab Execution Launcher

Self-staging launcher for the Visual / DSP prospective ablation study comparing:
1. `visual_control` (MobileNetV3 Frozen Backbone, 576-dim)
2. `dsp_only` (2D-FFT + 2D-DCT + Noise Residual + JPEG BAG, 16-dim)
3. `visual_dsp_fusion` (Concatenated 592-dim features)

All models share the same 5-outer x 4-inner source-grouped nested-CV folds across the 341 development sources (682 samples), the same Regularized Logistic Regression (L2, L-BFGS) classifier family, and the same inner-CV hyperparameter tuning. Zero access to the sealed Phase 4C.2G locked test.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

MODE = "pilot"  # "preflight", "pilot", or "full"
ALLOW_FULL = False  # Set to True only when ready to run all 15 outer fits
DEVICE = "cuda"     # "cuda" or "cpu" (visual feature extraction only; logistic regression runs on CPU)

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab")
CODE_ARCHIVE = DRIVE_ROOT / "visual_dsp_ablation/inputs/phase_visual_dsp_ablation_code.tar.gz"
DATASET_ARCHIVE = DRIVE_ROOT / "phase_4c1/inputs/phase_4c1_binary_n250_reusable.tar"

STAGE = Path("/content/visual_dsp_stage")
BUNDLE = Path("/content/development_bundle")
OUTPUT = DRIVE_ROOT / "visual_dsp_ablation/runs/execution_ablation"
REPORT_DIR = DRIVE_ROOT / "visual_dsp_ablation/report"


In [ ]:
import hashlib, importlib.metadata, json, os, platform, shutil, tarfile
from packaging.version import Version
from pathlib import PurePosixPath

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(archive_path, destination, mode):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tarfile.open(archive_path, mode) as archive:
        members = archive.getmembers()
        for member in members:
            posix = PurePosixPath(member.name)
            assert member.isfile(), f"Non-regular archive member: {member.name}"
            assert not posix.is_absolute() and ".." not in posix.parts and "\\" not in member.name
        archive.extractall(destination, members=members)

assert MODE in {"preflight", "pilot", "full"}
assert MODE != "full" or ALLOW_FULL, "Full 15-fit run requires ALLOW_FULL=True"
assert CODE_ARCHIVE.is_file(), f"Missing code archive: {CODE_ARCHIVE}"
assert DATASET_ARCHIVE.is_file(), f"Missing dataset archive: {DATASET_ARCHIVE}"

minimums = {"torch": "2.0.0", "numpy": "1.24.0", "scipy": "1.10.0", "Pillow": "9.5.0", "scikit-learn": "1.2.0", "PyYAML": "6.0"}
for pkg, min_ver in minimums.items():
    actual = importlib.metadata.version(pkg)
    assert Version(actual) >= Version(min_ver), f"{pkg} {actual} < {min_ver}"

safe_extract(CODE_ARCHIVE, STAGE, "r:gz")
safe_extract(DATASET_ARCHIVE, BUNDLE, "r:")

manifest = BUNDLE / "manifest_pilot_a_option_p.csv"
assert manifest.is_file(), "Manifest missing in extracted bundle"

OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Staged code and dataset successfully. Mode={MODE}, Output={OUTPUT}")


In [ ]:
import subprocess, sys

weights_path = STAGE / "models/research/pretrained/mobilenet_v3_large-8738ca79.pth"
if not weights_path.is_file():
    weights_path = DRIVE_ROOT / "phase_4c1/inputs/mobilenet_v3_large-8738ca79.pth"

cmd = [
    sys.executable, "-m", "ml.training.run_visual_dsp_ablation",
    "--mode", MODE,
    "--protocol", str(STAGE / "ml/configs/visual_dsp_ablation_protocol.yaml"),
    "--manifest", str(BUNDLE / "manifest_pilot_a_option_p.csv"),
    "--data-root", str(BUNDLE),
    "--weights-path", str(weights_path),
    "--output-dir", str(OUTPUT),
    "--device", DEVICE,
]
print("Executing runner:", " ".join(cmd))
subprocess.run(cmd, cwd=STAGE, check=True)


In [ ]:
if MODE == "full":
    cmd = [
        sys.executable, str(STAGE / "scripts/research/analyze_visual_dsp_ablation.py"),
        "--output-dir", str(OUTPUT),
        "--report-dir", str(REPORT_DIR),
    ]
    print("Running OOF Analysis:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    
    import pandas as pd
    metrics_df = pd.read_csv(REPORT_DIR / "oof_metrics.csv")
    print("=== OUT-OF-FOLD METRICS ===")
    display(metrics_df)
    
    deltas_df = pd.read_csv(REPORT_DIR / "paired_deltas.csv")
    print("\n=== PAIRED DELTAS ===")
    display(deltas_df)
else:
    receipts = list((OUTPUT / "fits").glob("*/outer_*/fit_receipt.json"))
    print(f"Completed {len(receipts)} fit(s) in {MODE} mode.")
    for r_path in sorted(receipts):
        receipt = json.loads(r_path.read_text())
        print(r_path.parent.name, receipt["recipe_id"], {
            "best_C": receipt["best_C"],
            "Macro-F1": round(receipt["test_metrics"]["macro_f1"], 4),
            "AUROC": round(receipt["test_metrics"]["auroc"], 4),
            "seconds": round(receipt["fit_wall_seconds"], 2),
        })
